<a href="https://colab.research.google.com/github/DanielWill-1/Capstone-Project---using-Hybrid-Vision-Transformers-for-Diabetic-Retinopathy-Staging/blob/main/aptos2019.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mariaherrerot/aptos2019")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'aptos2019' dataset.
Path to dataset files: /kaggle/input/aptos2019


In [ ]:
"""
Kaggle-ready Training script: Vision Transformer (ViT) binary classifier for APTOS 2019 (labels 0 or 4)

Dataset locations (predefined for Kaggle):
    TRAIN_CSV_PATH = '/kaggle/input/aptos2019/train_1.csv'
    TRAIN_IMAGES_DIR = '/kaggle/input/aptos2019/train_images/train_images'

How to use on Kaggle:
1. Add the APTOS 2019 dataset to your Kaggle notebook environment.
2. Run this script as a cell.
"""

import os
from pathlib import Path
from PIL import Image
import random

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split

# Dataset paths for Kaggle
TRAIN_CSV_PATH = '/kaggle/input/aptos2019/train_1.csv'
TRAIN_IMAGES_DIR = '/kaggle/input/aptos2019/train_images/train_images'
OUTPUT_DIR = '/kaggle/working/outputs'

# Configs
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 10
LR = 3e-5
WEIGHT_DECAY = 1e-2
VAL_FRAC = 0.2
SEED = 42
MODEL_NAME = 'vit_base_patch16_224'
NUM_WORKERS = 2


class AptosBinaryDataset(Dataset):
    def __init__(self, csv_file, img_dir, img_size=224, is_train=True):
        self.df = pd.read_csv(csv_file)
        if 'image' not in self.df.columns:
            if 'id_code' in self.df.columns:
                self.df = self.df.rename(columns={'id_code': 'image'})
            else:
                raise ValueError('CSV must contain `image` or `id_code` column')
        if 'diagnosis' not in self.df.columns:
            raise ValueError('CSV must contain `diagnosis` column')

        self.df['label'] = (self.df['diagnosis'] > 0).astype(int)

        self.img_dir = Path(img_dir)
        self.is_train = is_train
        self.img_size = img_size

        self.train_transform = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)], p=0.5),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        ])

        self.val_transform = transforms.Compose([
            transforms.Resize(int(img_size*1.14)),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.img_dir / row['image']
        if not img_path.exists():
            for ext in ['.png', '.jpg', '.jpeg', '.tif', '.tiff']:
                candidate = img_path.with_suffix(ext)
                if candidate.exists():
                    img_path = candidate
                    break

        img = Image.open(img_path).convert('RGB')
        transform = self.train_transform if self.is_train else self.val_transform
        img = transform(img)
        label = torch.tensor(row['label'], dtype=torch.long)
        return img, label


def split_csv(csv_file, val_frac=0.2, seed=42):
    df = pd.read_csv(csv_file)
    if 'image' not in df.columns and 'id_code' in df.columns:
        df = df.rename(columns={'id_code': 'image'})
    df['label'] = (df['diagnosis'] > 0).astype(int)
    train_df, val_df = train_test_split(df, test_size=val_frac, random_state=seed, stratify=df['label'])
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


def create_model(model_name='vit_base_patch16_224', pretrained=True, num_classes=1):
    model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
    return model


def train_one_epoch(model, dataloader, optimizer, criterion, device, scaler=None):
    model.train()
    running_loss = 0.0
    preds, targets = [], []
    for imgs, labels in dataloader:
        imgs = imgs.to(device)
        labels = labels.to(device).float().unsqueeze(1)

        optimizer.zero_grad()
        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(imgs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        probs = torch.sigmoid(logits).detach().cpu().numpy().ravel().tolist()
        preds.extend(probs)
        targets.extend(labels.detach().cpu().numpy().ravel().tolist())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_auc = roc_auc_score(targets, preds) if len(set(targets)) > 1 else float('nan')
    epoch_acc = accuracy_score(targets, [1 if p>=0.5 else 0 for p in preds])
    return epoch_loss, epoch_acc, epoch_auc


@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    preds, targets = [], []
    for imgs, labels in dataloader:
        imgs = imgs.to(device)
        labels = labels.to(device).float().unsqueeze(1)
        logits = model(imgs)
        loss = criterion(logits, labels)
        running_loss += loss.item() * imgs.size(0)
        probs = torch.sigmoid(logits).detach().cpu().numpy().ravel().tolist()
        preds.extend(probs)
        targets.extend(labels.detach().cpu().numpy().ravel().tolist())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_auc = roc_auc_score(targets, preds) if len(set(targets)) > 1 else float('nan')
    epoch_acc = accuracy_score(targets, [1 if p>=0.5 else 0 for p in preds])
    cm = confusion_matrix(targets, [1 if p>=0.5 else 0 for p in preds])
    return epoch_loss, epoch_acc, epoch_auc, cm


def main():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    train_df, val_df = split_csv(TRAIN_CSV_PATH, val_frac=VAL_FRAC, seed=SEED)
    train_csv = os.path.join(OUTPUT_DIR, 'train_split.csv')
    val_csv = os.path.join(OUTPUT_DIR, 'val_split.csv')
    train_df.to_csv(train_csv, index=False)
    val_df.to_csv(val_csv, index=False)

    train_ds = AptosBinaryDataset(train_csv, TRAIN_IMAGES_DIR, IMG_SIZE, is_train=True)
    val_ds = AptosBinaryDataset(val_csv, TRAIN_IMAGES_DIR, IMG_SIZE, is_train=False)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Using device:', device)

    model = create_model(MODEL_NAME, pretrained=True, num_classes=1)
    model.to(device)

    labels = train_df['label'].values
    pos, neg = (labels == 1).sum(), (labels == 0).sum()
    pos_weight = torch.tensor([neg/pos if pos > 0 else 1.0], dtype=torch.float).to(device)
    print(f'pos={pos} neg={neg} pos_weight={pos_weight.item():.4f}')

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

    best_val_auc = 0.0
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc, train_auc = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler)
        val_loss, val_acc, val_auc, cm = validate(model, val_loader, criterion, device)
        scheduler.step()

        print(f'Epoch {epoch}/{EPOCHS}')
        print(f'  Train loss: {train_loss:.4f} acc: {train_acc:.4f} auc: {train_auc:.4f}')
        print(f'  Val   loss: {val_loss:.4f} acc: {val_acc:.4f} auc: {val_auc:.4f}')
        print('  Confusion matrix:\n', cm)

        if not np.isnan(val_auc) and val_auc > best_val_auc:
            best_val_auc = val_auc
            ckpt_path = os.path.join(OUTPUT_DIR, f'best_{MODEL_NAME}_epoch{epoch}_auc{val_auc:.4f}.pth')
            torch.save({'model_state': model.state_dict(), 'epoch': epoch, 'val_auc': val_auc}, ckpt_path)
            print('  Saved best model to', ckpt_path)

    final_path = os.path.join(OUTPUT_DIR, f'final_{MODEL_NAME}.pth')
    torch.save({'model_state': model.state_dict(), 'epoch': EPOCHS}, final_path)
    print('Training complete. Final model saved to', final_path)


if __name__ == '__main__':
    main()


Using device: cuda
pos=1197 neg=1147 pos_weight=0.9582
Epoch 1/10
  Train loss: 0.2010 acc: 0.9189 auc: 0.9741
  Val   loss: 0.1783 acc: 0.9522 auc: 0.9858
  Confusion matrix:
 [[266  21]
 [  7 292]]
  Saved best model to /kaggle/working/outputs/best_vit_base_patch16_224_epoch1_auc0.9858.pth
Epoch 2/10
  Train loss: 0.0838 acc: 0.9744 auc: 0.9932
  Val   loss: 0.1190 acc: 0.9659 auc: 0.9911
  Confusion matrix:
 [[274  13]
 [  7 292]]
  Saved best model to /kaggle/working/outputs/best_vit_base_patch16_224_epoch2_auc0.9911.pth
Epoch 3/10
  Train loss: 0.0626 acc: 0.9808 auc: 0.9969
  Val   loss: 0.1579 acc: 0.9608 auc: 0.9884
  Confusion matrix:
 [[270  17]
 [  6 293]]
Epoch 4/10
  Train loss: 0.0663 acc: 0.9799 auc: 0.9960
  Val   loss: 0.1603 acc: 0.9625 auc: 0.9884
  Confusion matrix:
 [[276  11]
 [ 11 288]]
Epoch 5/10
  Train loss: 0.0583 acc: 0.9799 auc: 0.9973
  Val   loss: 0.1471 acc: 0.9522 auc: 0.9892
  Confusion matrix:
 [[281   6]
 [ 22 277]]
Epoch 6/10
  Train loss: 0.0369 ac

In [ ]:
import matplotlib.pyplot as plt

# Example: adjust according to your variables
epochs = range(1, len(train_loss) + 1)

plt.figure(figsize=(15,5))

# Loss
plt.subplot(1,3,1)
plt.plot(epochs, train_loss, label='Train Loss')
plt.plot(epochs, val_loss, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curve')
plt.legend()

# Accuracy
plt.subplot(1,3,2)
plt.plot(epochs, train_acc, label='Train Acc')
plt.plot(epochs, val_acc, label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy Curve')
plt.legend()

# AUC
plt.subplot(1,3,3)
plt.plot(epochs, val_auc, label='Val AUC', color='purple')
plt.xlabel('Epoch')
plt.ylabel('AUC')
plt.title('Validation AUC')
plt.legend()

plt.show()


NameError: name 'train_loss' is not defined

In [ ]:
import torch
from PIL import Image

# Load model checkpoint
model = ViTClassifier()  # same class as before
model.load_state_dict(torch.load("best_model.pth"))
model.eval().to(device)

# Single image inference
def predict_image(img_path):
    img = Image.open(img_path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])
    tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(tensor)
        prob = torch.sigmoid(output).item()
        pred = 1 if prob >= 0.5 else 0
    return pred, prob

# Example usage
img_path = "/kaggle/input/aptos2019/train_images/train_images/000c1434d8d7.png"
pred, prob = predict_image(img_path)
print(f"Prediction: {pred} (prob={prob:.4f})")


NameError: name 'ViTClassifier' is not defined

In [ ]:
!pip install torch==2.3.0 torchvision==0.18.1 --upgrade --quiet


ERROR: Cannot install torch==2.3.0 and torchvision==0.18.1 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [ ]:
!pip install torchvision==0.18.1


  Using cached torchvision-0.18.1-cp312-cp312-manylinux1_x86_64.whl.metadata (6.6 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x